In [1]:
pip install numpy bm3d opencv-python

Note: you may need to restart the kernel to use updated packages.


main関数


In [2]:
import os
import sys
import numpy as np
import scipy.io as sio
from sklearn.metrics import adjusted_rand_score

# 警告メッセージ用の関数（MATLABのwarningの代わり）
def scipy_warning(message):
    print(f"Warning: {message}", file=sys.stderr)

print('This code replicate some of the experiments in "Blind PRNU-based Image Clustering for Source Identification"\n')

# パスの追加（Pythonではインポートパスを通すか、自作モジュールとしてimportします）
# ※ 独自関数（ResidualInnerProductMatrix等）をPythonで実装し、'utility' や 'clustering' フォルダに配置したと仮定します。
# from utility import ResidualInnerProductMatrix, performanceIndexs, clusterView
# from clustering import TwoStepEnsembleClustering

ds_root = ''

if ds_root == '':
    scipy_warning('Please specify the path for the Dresden image database')

sets = [
    'A.1',
    # 'A.2',
    # 'A.max',
]

# %% Residual Inner Product Matrix (C) Parameters
denoising_function = 'bm3d'  # 'none'
crop_size = [np.inf, np.inf]  # [512, 512]
crop_location = 'center'      # 'upper-left'
in_memory = False             # True

# %% Two step ensemble Parameters
method_ensemble = 'WEAC-SL'
use_parfor = True              # 並列計算の有効化（Pythonではマルチプロセッシングなど）
verbose = True                 # 情報の表示
showCluster = True             # 結果をグラフで表示
showallclust = True            # 単一要素のクラスターも表示
putLegend = True               # グラフに凡例を表示

# ダミー関数（実際には独自実装が必要です）
def ResidualInnerProductMatrix(images_name, denoising_function, crop_size, crop_location, in_memory):
    # 本来はここで画像からPRNU（ノイズ）を抽出し、内積マトリクスを計算します
    raise NotImplementedError("PRNU抽出と内積計算のロジックをPythonで実装する必要があります。")

def TwoStepEnsembleClustering(C, method_ensemble, use_parfor, verbose):
    # 本来はここでWEAC-SLアルゴリズム（クラスタリング）を実行します
    raise NotImplementedError("WEAC-SLクラスタリングのロジックをPythonで実装する必要があります。")

def performance_indices(Index, gtruth):
    # ARIはsklearnで計算可能
    ari = adjusted_rand_score(gtruth, Index)

    # TPR, FPRの計算ロジック（MATLABの performanceIndexs に合わせる必要があります）
    # ※ここではダミーとして0を返しています
    tpr = 0.0
    fpr = 0.0
    return ari, tpr, fpr

def cluster_view(Index, gtruth, cameras, title, showallclust, put_legend):
    # matplotlibなどを使ってクラスターを可視化するロジック
    print(f"[Visualizing Graph]: {title}")


# メインループ
for exp_name in sets:
    print(f"Set: {exp_name}")

    # %% Preparing Experiments and Variables
    # MATLABの .mat ファイルを読み込む
    mat_path = f'./Data/{exp_name}.mat'
    if not os.path.exists(mat_path):
        print(f"Error: {mat_path} not found.")
        continue

    mat_data = sio.loadmat(mat_path)

    # mat_dataからデータを取り出す（型変換に注意が必要）
    images_name = mat_data['images_name']
    cameras = mat_data['cameras']
    gtruth = mat_data['gtruth'].flatten() # 1次元配列に平坦化

    # 画像パスの結合
    # images_nameがセルの配列の場合、Pythonではnumpyオブジェクトやリストとして処理します
    # images_name = [os.path.join(ds_root, str(img[0])) for img in images_name]

    print('Calculating Distances...')
    # C = ResidualInnerProductMatrix(images_name, denoising_function, crop_size, crop_location, in_memory)

    print('Clustering data...')
    # Index = TwoStepEnsembleClustering(C, method_ensemble, use_parfor, verbose)

    # --- 以下の出力確認用にダミーデータを設定（実際は上記関数から取得） ---
    Index = gtruth  # テスト用のダミー
    # -------------------------------------------------------------

    ari_2step, tpr_2step, fpr_2step = performance_indices(Index, gtruth)
    num_clusters = len(np.unique(Index))

    print('Clustering Performance------')
    print(f'ARI\t=\t {ari_2step:.3f}')
    print(f'TPR\t=\t {tpr_2step * 100:.2f}%')
    print(f'FPR\t=\t {fpr_2step * 100:.2f}%')
    print(f'N \t=\t {num_clusters}')
    print('----------------------------')

    if showCluster:
        title_str = f"Set {exp_name} - P = {ari_2step:.4f} - Nclust = {num_clusters}"
        cluster_view(Index, gtruth, cameras, title_str, showallclust, True)

This code replicate some of the experiments in "Blind PRNU-based Image Clustering for Source Identification"

Set: A.1
Error: ./Data/A.1.mat not found.


In [13]:
import numpy as np
import bm3d

def scipy_warning(message):
    import sys
    print(f"Warning: {message}", file=sys.stderr)

def my_bm3d(X, sigma):
    """
    MATLABの bm3d.m の挙動を再現する関数。
    入力X: NumPy配列 (H, W) または (H, W, C)、範囲は任意 (0-255等)
    """
    # 入力チェック
    if not isinstance(sigma, (int, float)):
        raise ValueError('"sigma" はスカラ値でなければなりません')

    # BM3Dライブラリは0〜1（あるいは0〜255）のfloat型、かつ2Dまたは3D配列を受け付けます
    # MATLAB版は内部で各チャンネルを独立して `actual_BM3D` にかけ、0-255の範囲にスケーリングしています
    X = np.atleast_3d(X).astype(np.float64)
    Nr, Nc, Nb = X.shape
    Y = np.zeros_like(X)

    # 低複雑度プロファイル 'lc' に相当する設定として、
    # bm3dライブラリの stage_arg（BM3DProfile）を調整することも可能ですが、
    # 通常はライブラリの標準最適化を利用します。
    # ここではMATLAB版同様、チャンネルごとにループ処理します。
    for i in range(Nb):
        channel = X[:, :, i]

        # 内部の actual_BM3D は [0, 1] への正規化を考慮しているため、
        # 入力が 0-255 の場合は、シグマを 255 で割って正規化空間で処理します
        max_val = np.max(channel)
        is_0_255 = max_val > 10  # MATLAB版の naive check (max(z(:)) > 10) を再現

        if is_0_255:
            channel_norm = channel / 255.0
            sigma_norm = sigma / 255.0
        else:
            channel_norm = channel
            sigma_norm = sigma

        # BM3Dによるノイズ除去を実行
        # bm3d.BM3D_PROFILE_FAST は MATLAB の 'lc' (Low Complexity) に近いです
        denoised = bm3d.bm3d(channel_norm, sigma_norm, profile='np')
        
        if is_0_255:
            Y[:, :, i] = denoised * 255.0
        else:
            Y[:, :, i] = denoised

    # 入力がもともと2次元（グレースケール）なら、2次元に絞って返す
    if Y.shape[2] == 1:
        Y = Y.squeeze(axis=2)

    return Y


def my_cbm3d(X, sigma):
    """
    MATLABの cbm3d.m の挙動を再現する関数。
    カラー画像の場合、輝度・色差空間（YUV / Opponent空間）に変換してノイズ除去を行います。
    """
    if not isinstance(sigma, (int, float)):
        raise ValueError('"sigma" はスカラ値でなければなりません')

    X = np.array(X, dtype=np.float64)

    if X.ndim == 3 and X.shape[2] == 3:
        # カラー画像の場合
        # MATLAB版の naive check を再現し、必要に応じて0-1に正規化
        max_val = np.max(X)
        is_0_255 = max_val > 10

        if is_0_255:
            X_norm = X / 255.0
            sigma_norm = sigma / 255.0
        else:
            X_norm = X
            sigma_norm = sigma

        # Pythonのbm3dライブラリには、カラー対応の `bm3d_rgb` が用意されており、
        # 内部で自動的に輝度・色差（Opponent）空間への変換と適切なノイズ除去を行ってくれます。
        denoised = bm3d.bm3d_rgb(X_norm, sigma_norm, profile='np')

        if is_0_255:
            return denoised * 255.0
        else:
            return denoised
    else:
        # グレースケール画像の場合は通常のbm3dを呼び出す
        return my_bm3d(X, sigma)


def bm3d_log(X, sigma):
    """
    MATLABの bm3d_log.m の挙動を完全再現する関数。
    """
    X = np.array(X, dtype=np.float64)
    eps = np.finfo(float).eps  # MATLABの `eps` に相当する微小値

    # 1. 対数変換（ドメイン変換）
    Xlog = np.log(X + eps)

    # 2. ノイズ平均の減算（MATLABコード内で mean_val = 0 なので実質変化なし）
    mean_val = 0
    Xlog = Xlog - mean_val

    # 3. 0-255 の範囲へスケーリング
    mi = np.min(Xlog)
    Mi = np.max(Xlog)

    # 万が一、分母が0になる（画像が完全に単色）場合の対策
    if Mi - mi == 0:
        Xlog_scaled = np.zeros_like(Xlog)
    else:
        Xlog_scaled = (255.0 / (Mi - mi)) * (Xlog - mi)

    # 4. ノイズ除去（上で定義した my_bm3d を呼び出す）
    Ylog = my_bm3d(Xlog_scaled, sigma)

    # MATLABコードの `Ylog = 255 * Ylog;` は、実際の実際上のバグ、
    # もしくは特殊な入力（[0,1]で戻ってきた場合）の補正処理です。
    # Pythonのbm3dは入力スケールを維持して返すため、
    # MATLABの `255 * Ylog` 自体はスケーリングを元に戻すステップ5と相殺する形で調整します。

    # 5. 元の範囲に逆スケーリング
    if Mi - mi != 0:
        Ylog = Ylog / (255.0 / (Mi - mi)) + mi

    # 6. 線形ドメインに逆変換（指数変換）
    Y = np.exp(Ylog) - eps

    return Y

# --- 使用例 ---
if __name__ == "__main__":
    # テスト用のダミー画像（128x128のカラー画像）を作成
    np.random.seed(0)
    img_pure = np.ones((128, 128, 3)) * 128
    noise = np.random.normal(0, 15, img_pure.shape)
    img_noisy = np.clip(img_pure + noise, 0, 255)

    print("元のノイズ画像形状:", img_noisy.shape)

    # 1. 通常のBM3D
    out_bm3d = my_bm3d(img_noisy, sigma=15)
    print("my_bm3d 完了")

    # 2. カラー用BM3D
    out_cbm3d = my_cbm3d(img_noisy, sigma=15)
    print("my_cbm3d 完了")

    # 3. 対数ドメインBM3D
    out_log = bm3d_log(img_noisy, sigma=15)
    print("bm3d_log 完了")

元のノイズ画像形状: (128, 128, 3)
my_bm3d 完了
my_cbm3d 完了
bm3d_log 完了


エネルギー関数？　❌

In [14]:
import numpy as np
import scipy.sparse as sp

# ※注意: 実際に動かすには QPBO のPythonラッパーライブラリが必要です。
# 例: pip install pyqpbo (環境によってはC++コンパイラの設定が必要になります)
try:
    import pyqpbo
except ImportError:
    print("Warning: pyqpbo がインストールされていません。ダミー関数で代用します。")

def CCEnergy(w, l):
    """
    相関クラスタリングのエネルギーを計算する関数（CCEnergy.m に相当）
    エネルギーが低い（マイナスに大きい）ほど、綺麗にグループ分けできている証拠です。

    Parameters:
        w (scipy.sparse.csr_matrix): 類似度のスパース行列（対角成分は0）
        l (numpy.ndarray): 各画像の現在のラベル（グループID）
    """
    # ゼロ以外の要素（線が引かれている画像ペア）のインデックスを取得
    ii, jj = w.nonzero()

    # 同じグループに属しているペアだけを抽出
    same_label = l[ii] == l[jj]

    # 同じグループ同士の線の太さ（重み）の合計をマイナスにする
    # （仲良しが同じグループにいるほどエネルギーが下がる）
    energy = -np.sum(w.data[same_label])

    return energy / 2.0  # 対称行列で2重カウントされるため半分にする

def BinaryExpand(w, l, li):
    """
    特定のグループ(li)の陣地を広げる最適化処理（BinaryExpand.m に相当）
    """
    nl = l.copy()

    # すでにラベル「li」を持っているノードは計算から除外
    non_li_idx = np.where(l != li)[0]
    n = len(non_li_idx)

    if n == 0:
        return nl  # すべてのノードがすでに「li」なら何もしない

    # --- 1. Unary項（1つのノード単体に対するコスト）の計算 ---
    # 0: 今のラベルを維持するコスト
    # 1: 新しくラベル「li」に乗り換えるコスト
    unary = np.zeros((n, 2), dtype=np.float64)

    # ノードが「li」に乗り換えた場合、既存の「li」ノードとのつながり（重み）が得られる
    is_li = (l == li)
    w_non_li = w[non_li_idx, :]
    # scipy.sparseの行列積を使って高速に計算
    unary[:, 1] = -np.array(w_non_li[:, is_li].sum(axis=1)).flatten()

    # --- 2. Pairwise項（ノード同士のつながりに対するコスト）の計算 ---
    # 対象ノード同士の部分行列を抽出（下三角行列のみ）
    w_sub = sp.tril(w[non_li_idx, :][:, non_li_idx], -1)
    ii, jj, wij = sp.find(w_sub)

    edges = np.vstack((ii, jj)).T.astype(np.int32)
    edge_weights = np.zeros((len(ii), 4), dtype=np.float64)

    rl = l[non_li_idx]
    old_differ = (rl[ii] != rl[jj]).astype(np.float64)

    # QPBOに渡すための4つの状態コスト (E00, E01, E10, E11)
    # E00: 両方とも今のラベルを維持
    # E01 / E10: 片方だけが「li」に乗り換える
    # E11: 両方とも「li」に乗り換える
    edge_weights[:, 0] = wij * old_differ  # E00
    edge_weights[:, 1] = wij               # E01
    edge_weights[:, 2] = wij               # E10
    edge_weights[:, 3] = 0.0               # E11 (両方liになるのでコスト0)

    # --- 3. QPBOアルゴリズムによる最適化 ---
    if 'pyqpbo' in globals():
        # pyqpbo を使ってグラフカットを解く (0: 維持, 1: 乗り換え)
        result_labels = pyqpbo.solve_qpbo(unary, edges, edge_weights)
        changed = (result_labels == 1)
        nl[non_li_idx[changed]] = li
    else:
        # ※ライブラリがない場合のダミー処理（実際にはここにQPBOが入ります）
        pass

    return nl

def a_expand(w, ig=None):
    """
    Alpha-Expansionのメインループ（a_expand.m に相当）
    """
    n = w.shape[0]

    # 対角成分（自分自身との類似度）を0にする
    w = w.copy()
    w.setdiag(0)
    w.eliminate_zeros()

    # 初期ラベル（ig）の準備
    if ig is not None and len(ig) == n:
        # 重複を省いて連番に変換（MATLABの unique 相当）
        _, l = np.unique(ig, return_inverse=True)
    else:
        l = np.ones(n, dtype=np.int32)

    NL = len(np.unique(l))
    current_energy = CCEnergy(w, l)

    while True:
        accepted = False
        li = 0

        while li < NL:
            # 各グループ（li）について、陣地を広げられるかテストする
            nl = BinaryExpand(w, l, li)
            new_energy = CCEnergy(w, nl)

            # エネルギーが下がった（より綺麗にグループ分けできた）場合、採用！
            if new_energy < current_energy:
                current_energy = new_energy
                l = nl
                accepted = True
                NL = np.max(nl) + 1  # 最大ラベルIDを更新

            li += 1

        # どのグループを広げようとしても改善しなくなったらループ終了
        if not accepted:
            break

        # 空になったグループを詰めて整理する
        _, l = np.unique(l, return_inverse=True)
        NL = len(np.unique(l))

    return l

比較表の作成（内積計算のゾーン）

In [16]:
import numpy as np
import cv2
import os
from tqdm import tqdm
# 先ほど作成した bm3d のラッパー関数をインポート（または同じファイルに記述）
# from bm3d_utils import my_bm3d

def get_crop_slice(img_shape, crop_size, crop_location):
    """
    画像の切り抜き（Crop）範囲を計算する補助関数
    """
    h, w = img_shape[:2]
    ch, cw = crop_size

    # サイズが Inf（無限大）に指定されている場合は切り抜かない
    if ch == np.inf or cw == np.inf:
        return slice(None), slice(None)

    ch, cw = int(ch), int(cw)

    if crop_location == 'center':
        y_start = (h - ch) // 2
        x_start = (w - cw) // 2
    elif crop_location == 'upper-left':
        y_start, x_start = 0, 0
    else:
        raise ValueError("crop_location は 'center' か 'upper-left' を指定してください")

    return slice(y_start, y_start + ch), slice(x_start, x_start + cw)

def extract_noise_residual(img_path, denoising_function, crop_size, crop_location, sigma=5):
    """
    1枚の画像からノイズ（残差 = 元画像 - 除去後画像）を抽出する補助関数
    """
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"画像が見つかりません: {img_path}")

    # PRNU（カメラ指紋）の抽出は一般的にグレースケールで行われます
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"画像の読み込みに失敗しました: {img_path}")

    img = img.astype(np.float64)

    # 1. フィルターで「被写体の景色」成分を作る
    if denoising_function == 'bm3d':
        # ※ここで先ほど作った my_bm3d を呼び出します
        denoised_img = my_bm3d(img, sigma=sigma)
    elif denoising_function == 'none':
        denoised_img = img
    else:
        raise NotImplementedError(f"未対応のフィルター: {denoising_function}")

    # 2. 残差（ノイズ）の計算: ノイズ = 元画像 - 景色
    residual = img - denoised_img

    # 3. 指定サイズに切り抜き
    sy, sx = get_crop_slice(residual.shape, crop_size, crop_location)
    residual_cropped = residual[sy, sx]

    # 4. ノイズの平均をゼロにする（比較計算の精度を上げるための一般的な前処理）
    residual_cropped = residual_cropped - np.mean(residual_cropped)

    return residual_cropped

def ResidualInnerProductMatrix(images_name, denoising_function='bm3d', crop_size=[np.inf, np.inf], crop_location='center', in_memory=True):
    """
    総当たりの内積マトリクス（C）を計算するメイン関数
    """
    num_images = len(images_name)
    C = np.eye(num_images)  # 対角成分が1（自分自身との比較）の単位行列を作成

    residuals = []

    print("Step 1/2: 各画像からノイズ（指紋）を抽出中...")
    for path in tqdm(images_name):
        res = extract_noise_residual(path, denoising_function, crop_size, crop_location)

        # 内積計算を高速化するため、あらかじめベクトル長を1に正規化（単位ベクトル化）しておく
        norm = np.linalg.norm(res)
        if norm > 0:
            res_normalized = res / norm
        else:
            res_normalized = np.zeros_like(res)

        residuals.append(res_normalized)

    print("Step 2/2: 総当たりで類似度（内積マトリクス）を計算中...")
    # 総当たり戦（組み合わせ）の計算
    for i in tqdm(range(num_images)):
        for j in range(i + 1, num_images):
            # 2つのノイズ成分を重ね合わせて、どれくらい一致するか（内積）を計算
            # np.vdot は平坦化して内積をとるため、2次元の画像マトリクス同士の比較に最適
            similarity = np.vdot(residuals[i], residuals[j])

            # Cマトリクスは対称行列（AとBの類似度 ＝ BとAの類似度）なので両方に代入
            C[i, j] = similarity
            C[j, i] = similarity

    return C

エネルギー関数によってのクラスタリング部分

In [17]:
import numpy as np
import scipy.sparse as sp

# ※注意: 実際に動かすには QPBO のPythonラッパーライブラリが必要です。
# 例: pip install pyqpbo (環境によってはC++コンパイラの設定が必要になります)
try:
    import pyqpbo
except ImportError:
    print("Warning: pyqpbo がインストールされていません。ダミー関数で代用します。")

def CCEnergy(w, l):
    """
    相関クラスタリングのエネルギーを計算する関数（CCEnergy.m に相当）
    エネルギーが低い（マイナスに大きい）ほど、綺麗にグループ分けできている証拠です。

    Parameters:
        w (scipy.sparse.csr_matrix): 類似度のスパース行列（対角成分は0）
        l (numpy.ndarray): 各画像の現在のラベル（グループID）
    """
    # ゼロ以外の要素（線が引かれている画像ペア）のインデックスを取得
    ii, jj = w.nonzero()

    # 同じグループに属しているペアだけを抽出
    same_label = l[ii] == l[jj]

    # 同じグループ同士の線の太さ（重み）の合計をマイナスにする
    # （仲良しが同じグループにいるほどエネルギーが下がる）
    energy = -np.sum(w.data[same_label])

    return energy / 2.0  # 対称行列で2重カウントされるため半分にする

def BinaryExpand(w, l, li):
    """
    特定のグループ(li)の陣地を広げる最適化処理（BinaryExpand.m に相当）
    """
    nl = l.copy()

    # すでにラベル「li」を持っているノードは計算から除外
    non_li_idx = np.where(l != li)[0]
    n = len(non_li_idx)

    if n == 0:
        return nl  # すべてのノードがすでに「li」なら何もしない

    # --- 1. Unary項（1つのノード単体に対するコスト）の計算 ---
    # 0: 今のラベルを維持するコスト
    # 1: 新しくラベル「li」に乗り換えるコスト
    unary = np.zeros((n, 2), dtype=np.float64)

    # ノードが「li」に乗り換えた場合、既存の「li」ノードとのつながり（重み）が得られる
    is_li = (l == li)
    w_non_li = w[non_li_idx, :]
    # scipy.sparseの行列積を使って高速に計算
    unary[:, 1] = -np.array(w_non_li[:, is_li].sum(axis=1)).flatten()

    # --- 2. Pairwise項（ノード同士のつながりに対するコスト）の計算 ---
    # 対象ノード同士の部分行列を抽出（下三角行列のみ）
    w_sub = sp.tril(w[non_li_idx, :][:, non_li_idx], -1)
    ii, jj, wij = sp.find(w_sub)

    edges = np.vstack((ii, jj)).T.astype(np.int32)
    edge_weights = np.zeros((len(ii), 4), dtype=np.float64)

    rl = l[non_li_idx]
    old_differ = (rl[ii] != rl[jj]).astype(np.float64)

    # QPBOに渡すための4つの状態コスト (E00, E01, E10, E11)
    # E00: 両方とも今のラベルを維持
    # E01 / E10: 片方だけが「li」に乗り換える
    # E11: 両方とも「li」に乗り換える
    edge_weights[:, 0] = wij * old_differ  # E00
    edge_weights[:, 1] = wij               # E01
    edge_weights[:, 2] = wij               # E10
    edge_weights[:, 3] = 0.0               # E11 (両方liになるのでコスト0)

    # --- 3. QPBOアルゴリズムによる最適化 ---
    if 'pyqpbo' in globals():
        # pyqpbo を使ってグラフカットを解く (0: 維持, 1: 乗り換え)
        result_labels = pyqpbo.solve_qpbo(unary, edges, edge_weights)
        changed = (result_labels == 1)
        nl[non_li_idx[changed]] = li
    else:
        # ※ライブラリがない場合のダミー処理（実際にはここにQPBOが入ります）
        pass

    return nl

def a_expand(w, ig=None):
    """
    Alpha-Expansionのメインループ（a_expand.m に相当）
    """
    n = w.shape[0]

    # 対角成分（自分自身との類似度）を0にする
    w = w.copy()
    w.setdiag(0)
    w.eliminate_zeros()

    # 初期ラベル（ig）の準備
    if ig is not None and len(ig) == n:
        # 重複を省いて連番に変換（MATLABの unique 相当）
        _, l = np.unique(ig, return_inverse=True)
    else:
        l = np.ones(n, dtype=np.int32)

    NL = len(np.unique(l))
    current_energy = CCEnergy(w, l)

    while True:
        accepted = False
        li = 0

        while li < NL:
            # 各グループ（li）について、陣地を広げられるかテストする
            nl = BinaryExpand(w, l, li)
            new_energy = CCEnergy(w, nl)

            # エネルギーが下がった（より綺麗にグループ分けできた）場合、採用！
            if new_energy < current_energy:
                current_energy = new_energy
                l = nl
                accepted = True
                NL = np.max(nl) + 1  # 最大ラベルIDを更新

            li += 1

        # どのグループを広げようとしても改善しなくなったらループ終了
        if not accepted:
            break

        # 空になったグループを詰めて整理する
        _, l = np.unique(l, return_inverse=True)
        NL = len(np.unique(l))

    return l

weacのクラスタリング部分

In [18]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

def weac_python(base_cls, ncai, cls_nums):
    """
    WEACアルゴリズムのPython実装
    base_cls: (N x M) 各列がクラスタリング結果
    ncai: (M,) 各クラスタリングの重み
    cls_nums: 分割したいクラスタ数のリスト
    """
    n_samples, n_base = base_cls.shape
    ncai = ncai.flatten()
    weight_sum = np.sum(ncai)

    # 1. 重み付き類似度行列の構築 (Co-association matrix)
    S = np.zeros((n_samples, n_samples))
    for k in range(n_base):
        # 各サンプルが同じクラスタに属しているかどうかの行列を作成
        labels = base_cls[:, k]
        # ブロードキャストで各ペアが同じクラスタか判定
        match = (labels[:, None] == labels[None, :])
        S += match.astype(float) * ncai[k]

    S = S / weight_sum

    # 2. 距離行列への変換 (similarity to distance: 1 - S)
    # scipyのlinkage関数は距離ベクトルを必要とするため
    dist_matrix = 1 - S
    dist_vector = squareform(dist_matrix, checks=False)

    # 3. 階層的クラスタリングの実行 (Average Linkageを使用)
    Z = linkage(dist_vector, method='average')

    # 4. 指定されたクラスタ数での分割
    results = {}
    for k in cls_nums:
        results[k] = fcluster(Z, t=k, criterion='maxclust')

    return results

# 使用例:
# ncai = getNCAI_python(base_cls) # ※getNCAIも同様に移植が必要
# consensus_results = weac_python(base_cls, ncai, [2, 3, 4])

リロード

In [11]:
import importlib
import clustering_utils
importlib.reload(clustering_utils)

from clustering_utils import (
    ResidualInnerProductMatrix,
    al_icm,
    weac_python,
    refine_clustering_labels
)

main関数

In [14]:
import os
import glob
import numpy as np

# これまで作成した各ステップの関数をインポートする想定
# ※実際の環境に合わせてファイル名や関数名を調整してください
from clustering_utils import (
    ResidualInnerProductMatrix,  # Step 1用
    al_icm,                      # Step 2用 (エネルギー最適化)
    weac_python,                 # Step 2用 (アンサンブル統合)
    refine_clustering_labels     # Step 3用 (洗練ステップ)
)

def main():
    print("=== Blind PRNU-based Image Clustering Pipeline ===")

    # ---------------------------------------------------------
    # [準備] 画像データの読み込み
    # ---------------------------------------------------------
    # 識別したい画像が入っているディレクトリを指定
    dataset_dir = "/Users/qingfengliu/Desktop/fashion-pic"  # ここを適宜変更してください
    images_name = sorted(glob.glob(os.path.join(dataset_dir, "*.jpg")) + glob.glob(os.path.join(dataset_dir, "*.png")))

    n_samples = len(images_name)
    if n_samples == 0:
        print("エラー: 指定されたディレクトリに画像が見つかりません。")
        return

    print(f"対象画像数: {n_samples}枚")

    # =========================================================
    # Step 1: 特徴抽出と内積マトリクスの計算
    # =========================================================
    print("\n--- Step 1: ノイズ残差の抽出と内積マトリクス(C)の計算 ---")
    # BM3Dなどのノイズ除去フィルタを使ってPRNUを抽出し、総当たりの類似度を計算
    # 例: すべての画像から中央 500x500 像素を切り出して計算する場合
    C = ResidualInnerProductMatrix(
        images_name, 
        denoising_function='bm3d',
        crop_size=(500, 500)  # または入力画像の中で最も小さいサイズ以下の数値
    )

    # =========================================================
    # Step 2: 初期クラスタリング (エネルギー最適化 + WEAC)
    # =========================================================
    print("\n--- Step 2: 初期クラスタリング (AL-ICM & WEAC) ---")

    # AL-ICM（最適化）のために、類似度行列をゼロ中心にシフトして反発力を作る
    threshold = np.mean(C)
    W = C - threshold

    # WEACに入力するための「複数の異なるクラスタリング結果（ベースクラスタ）」を作成
    print("複数のベースクラスタリングを生成中...")
    base_cls_list = []
    for seed in range(5):  # 例として5パターンの初期値から最適化
        np.random.seed(seed)
        random_init = np.random.randint(0, 3, n_samples)

        # 内積グラフWに対してエネルギー最適化を実行し、整合性のあるクラスタを得る
        opt_labels = al_icm(W, initial_labels=random_init)
        base_cls_list.append(opt_labels)

    base_cls = np.column_stack(base_cls_list)

    # ベースクラスタ群をWEACで1つに統合する
    # ※ダミーのNCAI（重み）として全て1を設定（本来は getNCAI 等で計算）
    dummy_ncai = np.ones(base_cls.shape[1])

    print("WEACによるアンサンブル統合を実行中...")
    # 想定されるカメラ（クラスタ）数が3だと仮定して実行
    weac_results = weac_python(base_cls, dummy_ncai, cls_nums=[3])
    initial_labels = weac_results[20]

    # =========================================================
    # Step 3: 洗練ステップ (Iterative Refinement)
    # =========================================================
    print("\n--- Step 3: 洗練ステップ (Iterative Refinement) ---")
    # 論文の手法に基づき、画像同士の生の内積値(C)を使って境界のラベルを微調整
    final_labels = refine_clustering_labels(C, initial_labels, max_iter=20)

    # =========================================================
    # 最終結果の出力
    # =========================================================
    print("\n=== 処理完了: 最終クラスタリング結果 ===")

    # カメラごとに画像をグループ分けして表示
    unique_cameras = np.unique(final_labels)
    for cam_id in unique_cameras:
        print(f"\n[カメラグループ {cam_id}]")
        for i, path in enumerate(images_name):
            if final_labels[i] == cam_id:
                print(f"  - {os.path.basename(path)}")

if __name__ == "__main__":
    main()

=== Blind PRNU-based Image Clustering Pipeline ===
対象画像数: 501枚

--- Step 1: ノイズ残差の抽出と内積マトリクス(C)の計算 ---
Step 1/2: 各画像からノイズ（指紋）を抽出中...


100%|██████████| 501/501 [1:48:50<00:00, 13.04s/it]


Step 2/2: 総当たりで類似度（内積マトリクス）を計算中...


100%|██████████| 501/501 [01:53<00:00,  4.41it/s]



--- Step 2: 初期クラスタリング (AL-ICM & WEAC) ---
複数のベースクラスタリングを生成中...
WEACによるアンサンブル統合を実行中...

--- Step 3: 洗練ステップ (Iterative Refinement) ---

=== 処理完了: 最終クラスタリング結果 ===

[カメラグループ 1]
  - 1638714174_1000.jpg
  - 1638803985_1000.jpg
  - 1639798168_1000.jpg
  - 1641268510_1000.jpg
  - 1643425890_1000.jpg
  - 1644742775_1000.jpg
  - 1644858775_1000.jpg
  - 1670425965_1000.jpg
  - 1672898298_1000.jpg
  - 1674901848_1000.jpg
  - 1675098267_1000.jpg
  - 1675834208_1000.jpg
  - 1677159894_1000.jpg
  - 1677509212_1000.jpg
  - 1702448491_1000.jpg
  - 1702612849_1000.jpg
  - 1703004241_1000.jpg
  - 1703774589_1000.jpg
  - 1704460707_1000.jpg
  - 1704613400_1000.jpg
  - 1704839374_1000.jpg
  - 1705856022_1000.jpg
  - 1706076306_1000.jpg
  - 1706448945_1000.jpg
  - 1706452599_1000.jpg
  - 1706798585_1000.jpg
  - 1707211318_1000.jpg
  - 1707219224_1000.jpg
  - 1708742695_1000.jpg
  - 1708784585_1000.jpg
  - 1731558915_1000.jpg
  - 1733285093_1000.jpg
  - 1733484749_1000.jpg
  - 1733875802_1000.jpg
  - 173401

それぞれのクラスタ数を保存するmainコード

In [ ]:
import os
import glob
import csv
import numpy as np

from clustering_utils import (
    ResidualInnerProductMatrix,  # Step 1用
    al_icm,                      # Step 2用 (エネルギー最適化)
    weac_python,                 # Step 2用 (アンサンブル統合)
    refine_clustering_labels     # Step 3用 (洗練ステップ)
)

def main():
    print("=== Blind PRNU-based Image Clustering Pipeline ===")

    # ---------------------------------------------------------
    # [準備] 画像データの読み込み
    # ---------------------------------------------------------
    dataset_dir = "/Users/qingfengliu/Desktop/fashion-pic"
    images_name = sorted(glob.glob(os.path.join(dataset_dir, "*.jpg")) + glob.glob(os.path.join(dataset_dir, "*.png")))

    n_samples = len(images_name)
    if n_samples == 0:
        print("エラー: 指定されたディレクトリに画像が見つかりません。")
        return

    print(f"対象画像数: {n_samples}枚")

    # 結果保存用フォルダの作成
    output_dir = "./clustering_output_results"
    os.makedirs(output_dir, exist_ok=True)

    # =========================================================
    # Step 1: 特徴抽出と内積マトリクスの計算 (1度のみ実行)
    # =========================================================
    print("\n--- Step 1: ノイズ残差の抽出と内積マトリクス(C)の計算 ---")
    C = ResidualInnerProductMatrix(
        images_name, 
        denoising_function='bm3d',
        crop_size=(500, 500)
    )

    # 類似度行列をゼロ中心にシフト
    threshold = np.mean(C)
    W = C - threshold

    # 試行するクラスタ数のリスト
    target_k_list = [5, 10, 15, 20]

    # =========================================================
    # Step 2 & 3: 各クラスタ数ごとの処理と保存
    # =========================================================
    for k in target_k_list:
        print(f"\n==========================================")
        print(f"  クラスタ数 k = {k} の処理を開始")
        print(f"==========================================")

        # -----------------------------------------------------
        # Step 2: 初期クラスタリング (AL-ICM & WEAC)
        # -----------------------------------------------------
        print(f"--- Step 2 (k={k}): 初期クラスタリング (AL-ICM & WEAC) ---")
        base_cls_list = []
        for seed in range(5):
            np.random.seed(seed)
            random_init = np.random.randint(0, k, n_samples)

            opt_labels = al_icm(W, initial_labels=random_init)
            base_cls_list.append(opt_labels)

        base_cls = np.column_stack(base_cls_list)
        dummy_ncai = np.ones(base_cls.shape[1])

        # WEACによるアンサンブル統合
        weac_results = weac_python(base_cls, dummy_ncai, cls_nums=[k])
        
        # WEACの返り値から該当するkの初期ラベルを取得
        if isinstance(weac_results, dict) and k in weac_results:
            initial_labels = weac_results[k]
        elif isinstance(weac_results, dict) and len(weac_results) > 0:
            initial_labels = list(weac_results.values())[0]
        else:
            initial_labels = weac_results

        # -----------------------------------------------------
        # Step 3: 洗練ステップ
        # -----------------------------------------------------
        print(f"--- Step 3 (k={k}): 洗練ステップ (Iterative Refinement) ---")
        final_labels = refine_clustering_labels(C, initial_labels, max_iter=20)

        # -----------------------------------------------------
        # 結果の出力 & ファイル保存
        # -----------------------------------------------------
        # 1. コンソール表示
        unique_cameras = np.unique(final_labels)
        print(f"\n[結果: k={k} のグループ分け (検出されたグループ数: {len(unique_cameras)})]")
        
        for cam_id in unique_cameras:
            assigned_files = [os.path.basename(images_name[i]) for i in range(n_samples) if final_labels[i] == cam_id]
            print(f"  カメラグループ {cam_id} ({len(assigned_files)}枚): {assigned_files[:3]}...")

        # 2. CSVファイルへの保存 (画像パス, クラスタID)
        csv_filename = os.path.join(output_dir, f"clustering_k{k}.csv")
        with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["image_path", "filename", "cluster_id"])
            for path, label in zip(images_name, final_labels):
                writer.writerow([path, os.path.basename(path), label])

        # 3. テキストファイルへの保存 (グループごとのリスト形式)
        txt_filename = os.path.join(output_dir, f"clustering_k{k}.txt")
        with open(txt_filename, mode="w", encoding="utf-8") as f:
            f.write(f"=== PRNU Clustering Results (Target k = {k}) ===\n")
            for cam_id in unique_cameras:
                f.write(f"\n[カメラグループ {cam_id}]\n")
                for i, path in enumerate(images_name):
                    if final_labels[i] == cam_id:
                        f.write(f"  - {os.path.basename(path)}\n")

        print(f"保存完了: {csv_filename}")
        print(f"保存完了: {txt_filename}")

    print("\n=== すべてのクラスタ数での処理が完了しました ===")

if __name__ == "__main__":
    main()